In [2]:
# Create a  Weather Tool MCP Server that any AI agent can use with sample use case

# 🌦️ MCP WEATHER TOOL + GROQ + GRADIO UI

# Install dependencies
!pip install groq gradio

# IMPORTS
import json
import gradio as gr
from groq import Groq
from google.colab import userdata

# Load API Key
client = Groq(api_key=userdata.get("GROQ_API_KEY"))

# STEP 1: WEATHER TOOL SERVER
class WeatherServer:
    def __init__(self):
        # Sample data (can replace with real API)
        self.weather_data = {
            "delhi": {"temp": 32, "condition": "Sunny"},
            "mumbai": {"temp": 28, "condition": "Humid"},
            "bangalore": {"temp": 25, "condition": "Cloudy"},
            "london": {"temp": 15, "condition": "Rainy"}
        }

    def get_weather(self, city):
        city = city.lower()

        if city in self.weather_data:
            data = self.weather_data[city]
            return {
                "status": "success",
                "city": city,
                "temperature": data["temp"],
                "condition": data["condition"]
            }

        return {"status": "error", "message": "City not found"}


# STEP 2: MCP AGENT (AI POWERED)
class MCPWeatherAgent:
    def __init__(self, server):
        self.server = server

    # 🧠 Decide if tool needed
    def should_call_tool(self, query):

        prompt = f"""
        Does this query need weather data?
        Answer only YES or NO.

        Query: {query}
        """

        res = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}]
        )

        decision = res.choices[0].message.content.lower()
        return "yes" in decision

    # 🧠 Extract city using AI
    def extract_city(self, query):

        prompt = f"""
        Extract city name from the query.
        Return only city name (one word).

        Query: {query}
        """

        res = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}]
        )

        return res.choices[0].message.content.strip().lower()

    # 🤖 Main MCP Flow
    def handle_query(self, query):

        if self.should_call_tool(query):

            city = self.extract_city(query)

            result = self.server.get_weather(city)

            if result["status"] == "success":
                return f"""
🌦️ Weather Report

📍 City: {result['city'].title()}
🌡️ Temperature: {result['temperature']}°C
☁️ Condition: {result['condition']}
"""
            else:
                return "❌ City not found. Try Delhi, Mumbai, Bangalore, London."

        else:
            # General AI response
            res = client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[{"role": "user", "content": query}]
            )

            return "🤖 " + res.choices[0].message.content


# STEP 3: CONNECT EVERYTHING
server = WeatherServer()
agent = MCPWeatherAgent(server)

def chatbot_fn(user_input):
    return agent.handle_query(user_input)


# STEP 4: GRADIO UI
interface = gr.Interface(
    fn=chatbot_fn,
    inputs=gr.Textbox(placeholder="Ask weather like: Weather in Delhi"),
    outputs="text",
    title="🌦️ MCP Weather Assistant (Groq AI)",
    description="AI-powered weather assistant using MCP + Groq + Gradio"
)

# Launch UI
interface.launch()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 4.7 MB/s eta 0:00:00
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ddde1bb62b4d3ab599.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
